Start with 12 on 6/23/2026

## 14. Literature-guided synthetic data generation
6/28/2026

In [1]:
import json

In [2]:
# import importlib
# import microbiome_eval.tools.pubmed_search

# importlib.reload(microbiome_eval.tools.pubmed_search)
# import microbiome_eval.tools.pubmed_search
from microbiome_eval.tools.pubmed_search import PubMed_Search_Tool

In [3]:
search_tool = PubMed_Search_Tool("none")

In [38]:
disease_terms = "(" + " OR ".join([
    "Chronic recurrent multifocal osteomyelitis",
    "CRMO",
    "Chronic multifocal osteomyelitis",
    "Multifocal osteomyelitis, chronic",
    "Osteomyelitis, Chronic Multifocal"
  ]) + ")"
disease_terms
# search_tool.run('query': "")

'(Chronic recurrent multifocal osteomyelitis OR CRMO OR Chronic multifocal osteomyelitis OR Multifocal osteomyelitis, chronic OR Osteomyelitis, Chronic Multifocal)'

In [44]:
# search_tool.run(**{'query': diseae_terms + " AND " + "Faecalibacterium AND microbiome"})
outputs = []
for disease in [
    "Chronic recurrent multifocal osteomyelitis",
    "CRMO",
    "Chronic multifocal osteomyelitis",
    "Multifocal osteomyelitis, chronic",
    "Osteomyelitis, Chronic Multifocal"
  ]:
    outputs.append(search_tool.run(**{'query': f'"{disease}"' + " AND " + "Faecalibacterium AND microbiome"}))

In [45]:
outputs

[{'All Journals': {'term_used': 'Chronic recurrent multifocal osteomyelitis AND Faecalibacterium AND microbiome',
   'summary': 'No publications found. Please try again with different keywords or journal.',
   'citations': []}},
 {'All Journals': {'term_used': 'CRMO AND Faecalibacterium AND microbiome',
   'summary': 'No publications found. Please try again with different keywords or journal.',
   'citations': []}},
 {'All Journals': {'term_used': 'Chronic multifocal osteomyelitis AND Faecalibacterium AND microbiome',
   'summary': 'No publications found. Please try again with different keywords or journal.',
   'citations': []}},
 {'All Journals': {'term_used': 'Multifocal osteomyelitis AND chronic AND Faecalibacterium AND microbiome',
   'summary': 'No publications found. Please try again with different keywords or journal.',
   'citations': []}},
 {'All Journals': {'term_used': 'Osteomyelitis AND Chronic Multifocal AND Faecalibacterium AND microbiome',
   'summary': 'No publications

In [10]:
queries = [
    {'query': "\"Graves Disease\" AND Faecalibacterium AND microbiome"}
]

for example in queries:
    result = search_tool.run(**example)
    break

In [11]:
result

{'All Journals': {'term_used': 'Graves Disease AND Faecalibacterium AND microbiome',
  'publications': [{'Title': "Faecalibacterium prausnitzii alleviates Graves' orbitopathy by modulating short-chain fatty acid (SCFA) metabolism and inhibiting orbital fibrosis and adipogenesis.",
    'Authors': ['Qi X', 'Zhang J', 'Li Y', 'Shi Y', 'Yao J', 'Tong B'],
    'Abstract': "Graves' orbitopathy (GO) is an autoimmune inflammatory disorder characterized by orbital fibrosis and adipose tissue expansion, leading to proptosis and ocular dysfunction. Recent evidence suggests that gut microbiota dysbiosis contributes to GO pathogenesis, particularly through alterations in short-chain fatty acid (SCFA) metabolism. This study investigated the role of Faecalibacterium prausnitzii (F. prausnitzii), a key butyrate-producing bacterium, in modulating GO-related fibrosis and adipogenesis. Clinical fecal sample analysis revealed a significant reduction in F. prausnitzii abundance in GO patients. In a murine 

In [ ]:
prompt_template = """
Based on the following scientific paper abstracts, what are some pathways or mechanisms influencing the effect of the bacteria {bacteria_name} on the disease {disease_name}? Please provide a concise summary of the key findings and mechanisms discussed in the abstracts. If no relevant information is found, please state "No relevant information found."

{abstracts}
"""

def format_publications(result):
    formatted_publications = ""
    for i, pub in enumerate(result['All Journals']['publications']):
        text = f"""
    Publication Abstract ([{i+1}] PMID: {pub['PMID']}):
    Title: {pub['Title']}
    {pub['Abstract']}
    """
        formatted_publications += text
    return formatted_publications

prompt = [{
    "role": "user",
    "content": prompt_template.format(
        bacteria_name="Faecalibacterium",
        disease_name="Graves Disease",
        abstracts=format_publications(result)
    )
}]

In [19]:
from microbiome_eval.llm import LLM

model = LLM("google/gemma-4-31B-it")

In [20]:
debug_response = model.call(prompt)

print(debug_response['content'])

Based on the provided abstracts, the bacteria *Faecalibacterium* (specifically *F. prausnitzii*) influences Graves' Disease (GD) and its complications through the following pathways and mechanisms:

**1. Modulation of Short-Chain Fatty Acid (SCFA) Metabolism**
*   **Butyrate Production:** *F. prausnitzii* is identified as a key butyrate-producing bacterium. Patients with GD and Graves' orbitopathy (GO) exhibit a significant reduction in *Faecalibacterium* abundance and a corresponding depletion of butyrate levels [1, 4, 5, 6].
*   **Anti-Fibrotic and Anti-Adipogenic Effects:** In the context of Graves' orbitopathy (GO), the restoration of butyrate levels via *F. prausnitzii* administration inhibits the activation of orbital fibroblasts. Specifically, it downregulates the expression of **$\alpha$-SMA, collagen I, PPAR$\gamma$, and C/EBP$\alpha$**, thereby suppressing orbital fibrosis (collagen deposition) and adipogenesis (lipid accumulation) [1].

**2. Systemic and Autoimmune Influence

In [22]:
prompt_template = """
Based on the following scientific paper abstracts, trace through the pathways or mechanisms that mediate the effect of the bacteria {bacteria_name} on the disease {disease_name}. 
Include a detailed reasoning trace, with a step-by-step explanation. Where information is missing, please succinctly indicate gaps in knowledge.
If no relevant information is found, please state "No relevant information found."

{abstracts}
"""


formatted_publications = ""
for i, pub in enumerate(result['All Journals']['publications']):
    text = f"""
Publication Abstract ([{i+1}] PMID: {pub['PMID']}):
Title: {pub['Title']}
{pub['Abstract']}
"""
    formatted_publications += text

prompt = [{
    "role": "user",
    "content": prompt_template.format(
        bacteria_name="Faecalibacterium",
        disease_name="Graves Disease",
        abstracts=formatted_publications
    )
}]

debug_response = model.call(prompt, temperature=1.0, top_p=0.95, top_k=64, chat_template_kwargs={"enable_thinking": True})
print(debug_response['content'])

Based on the provided abstracts, the pathways and mechanisms through which the bacteria *Faecalibacterium* (specifically *Faecalibacterium prausnitzii*) mediate effects on Graves' Disease (GD) and Graves' Orbitopathy (GO) are traced as follows:

### Reasoning Trace

**Step 1: Correlation of *Faecalibacterium* abundance with disease state**
Multiple studies establish a consistent correlation between the reduction of *Faecalibacterium* and the presence of autoimmune thyroid diseases. 
*   Clinical analysis shows a significant reduction of *Faecalibacterium* in patients with Graves' Disease [4, 5, 6] and specifically in those with Graves' Orbitopathy (GO) [1].
*   Conversely, *Faecalibacterium* is associated with healthy control groups [4].

**Step 2: Metabolic Output (The SCFA Pathway)**
The primary mechanism by which *F. prausnitzii* exerts its effect is through the production of short-chain fatty acids (SCFAs), specifically **butyrate** [1]. In patients with GO, butyrate levels are sig

This looks ok from a certain perspective. Unfortunately, GO is not the same as GD, but it's ok for now.
We'll refine this pipeline later to filter out abstracts.




In [24]:
# let's try with the Eubacteria
bacteria_name = "Eubacterium"
result = search_tool.run(**{'query': f"\"Graves Disease\" AND {bacteria_name} AND microbiome"})

prompt = [{
    "role": "user",
    "content": prompt_template.format(
        bacteria_name=bacteria_name,
        disease_name="Graves Disease",
        abstracts=format_publications(result)
    )
}]

In [26]:
print(format_publications(result))


    Publication Abstract ([1] PMID: 39336141):
    Title: Uncovering a Causal Connection between Gut Microbiota and Six Thyroid Diseases: A Two-Sample Mendelian Randomization Study.
    BACKGROUND: Recent studies have established associations between the gut microbiota (GM) and thyroid diseases (TDs). However, their causal relationships remain elusive. METHODS: To investigate this causality, we conducted a two-sample Mendelian randomization (MR) analysis using genome-wide association study (GWAS) data from MiBioGen and FinnGen, with GM as the exposure and six TDs as outcomes. RESULTS: We identified 32 microbial taxa linked to the risk of six TDs. The Clostridium innocuum group, Ruminiclostridium5, and Lachnoclostridium exhibited protective effects against nontoxic diffuse goiter (NDG). Conversely, an increased risk of NDG was associated with Ruminococcaceae UCG002, Alistipes, Methanobrevibacter, Marvinbryantia, and Ruminococcaceae UCG014. Bifidobacterium and Sutterella were protective

In [27]:
debug_response_eubacteria = model.call(prompt, temperature=1.0, top_p=0.95, top_k=64, chat_template_kwargs={"enable_thinking": True})
print(debug_response_eubacteria['content'])

Based on the provided abstracts, there is evidence of an association between various *Eubacterium* groups and Graves' Disease (GD), although **no specific biological pathways or molecular mechanisms** are detailed.

The key findings regarding *Eubacterium* and Graves' Disease are as follows:

*   **Risk Factors:** One study using Mendelian randomization identified the **_Eubacterium rectale_ group** as a risk factor for thyrotoxicosis with Graves' Disease [1]. Another study identified the **_Eubacterium brachy_ group** as having a significant association with the onset of Graves' Disease, suggesting a potential causal effect [3].
*   **Reduced Abundance:** In a cross-sectional study, patients with Graves' Disease exhibited a significantly lower abundance of the **_Eubacterium hallii_ group** compared to healthy controls [4].

While the abstracts suggest that gut microbiota act as "intrinsic regulators of thyroid autoimmunity" [4] and that causal relationships exist at the genetic level

In [ ]:
combination_prompt_template = """
A sample of the microbiome was taken from a patient with the disease {disease_name}, and the following bacteria were identified in the sample:

{abundance_table}

---
Using the mechanism descriptions below, simulate a reasoning process that starts with the the bacterial profile and ends up concluding what disease the patient has.
Your response should include a step-by-step reasoning trace, highlighting potential interactions between bacteria and their mechanisms. The relevant pathways for each of the bacteria types are highlighted below:

{pathways}
---

Using the bacteria profile and the pathway descriptions, simulating a reasoning process that starts with the profile and ends with concluding what disease the patient has ({disease_name}). This should be formatted as a step-by-step reasoning trace.
Do NOT include any citations to papers or explicitly mention the pathway descriptions in your answer.
""".strip()


formatted_pathways = ""
for disease, pathway_descr in zip(["Faecalibacterium", "Eubacterium"], [debug_response['content'], debug_response_eubacteria['content']]):
    formatted_pathways += f"\n## Pathways for {disease}:\n{pathway_descr}\n"
formatted_pathways = formatted_pathways.strip()

combination_prompt = [{
    "role": "user",
    "content": combination_prompt_template.format(
        disease_name="Graves Disease",
        abundance_table="Eubacterium: 66%\nFaecalibacterium: 33%\n",
        pathways=formatted_pathways
    )
}]

print(combination_prompt[0]['content'])


A sample of the microbiome was taken from a patient with the disease Graves Disease, and the following bacteria were identified in the sample:

Eubacterium: 66%
Faecalibacterium: 33%


---
Using the mechanism descriptions below, simulate a reasoning process that starts with the the bacterial profile and ends up concluding what disease the patient has.
Your response should include a step-by-step reasoning trace, highlighting potential interactions between bacteria and their mechanisms. The relevant pathways for each of the bacteria types are highlighted below:

## Pathways for Faecalibacterium:
Based on the provided abstracts, the pathways and mechanisms through which the bacteria *Faecalibacterium* (specifically *Faecalibacterium prausnitzii*) mediate effects on Graves' Disease (GD) and Graves' Orbitopathy (GO) are traced as follows:

### Reasoning Trace

**Step 1: Correlation of *Faecalibacterium* abundance with disease state**
Multiple studies establish a consistent correlation betwe

In [32]:
combination_resp = model.call(combination_prompt, temperature=1.0, top_p=0.95, top_k=64, chat_template_kwargs={"enable_thinking": False})
print(combination_resp['content'])



**Step 1: Analysis of Bacterial Profile**
The patient's microbiome sample shows a composition of 66% *Eubacterium* and 33% *Faecalibacterium*. To determine the clinical significance of this profile, the abundance of these specific taxa must be evaluated against known disease associations.

**Step 2: Evaluation of *Faecalibacterium* Levels**
The presence of *Faecalibacterium* at 33% is analyzed. In healthy individuals, this bacterium is typically more abundant. A reduction in *Faecalibacterium* triggers a specific metabolic cascade:
*   **Metabolic Deficiency:** Lower levels of *Faecalibacterium* lead to a decrease in the production of the short-chain fatty acid **butyrate**.
*   **Cellular Impact:** A deficiency in butyrate removes the inhibitory pressure on orbital fibroblasts. This leads to the upregulation of **$\alpha$-SMA** and **collagen I** (promoting fibrosis) and the upregulation of **PPAR$\gamma$** and **C/EBP$\alpha$** (promoting adipogenesis).
*   **Physiological Outcome:**

In [33]:
combination_resp = model.call(combination_prompt, temperature=1.0, top_p=0.95, top_k=64, chat_template_kwargs={"enable_thinking": True})
print(combination_resp['content'])

**Step 1: Analysis of *Faecalibacterium* Abundance**
The microbiome profile shows *Faecalibacterium* at 33%. This indicates a reduction in abundance, which is a consistent clinical marker for autoimmune thyroid conditions.

**Step 2: Metabolic Impact of *Faecalibacterium* Reduction**
The decrease in *Faecalibacterium* leads to a proportional decline in the production of short-chain fatty acids (SCFAs), specifically butyrate.

**Step 3: Impact on Orbital Fibroblasts**
A deficiency in butyrate removes a critical inhibitory signal for orbital fibroblasts. This results in:
*   **Increased Fibrosis:** An upregulation of $\alpha$-SMA and collagen I, leading to excessive collagen deposition.
*   **Increased Adipogenesis:** An upregulation of PPAR$\gamma$ and C/EBP$\alpha$, which triggers the differentiation of fibroblasts into adipocytes, causing lipid accumulation.

**Step 4: Clinical Physiological Manifestations**
The combined increase in fibrosis and adipogenesis within the orbital tissues

I kind of like the non-thinking one a bit better in this case...mostly because it includes more of the details

In [ ]:
# query-expander

prompt_template = """
We want to search for scientific papers that discuss the relationship betwen the bacteria {bacteria_name} in the gut microbiome and the disease {disease_name}.
""".strip()


Publication Abstract ([1] PMID: 40812530):
Title: Faecalibacterium prausnitzii alleviates Graves' orbitopathy by modulating short-chain fatty acid (SCFA) metabolism and inhibiting orbital fibrosis and adipogenesis.
Graves' orbitopathy (GO) is an autoimmune inflammatory disorder characterized by orbital fibrosis and adipose tissue expansion, leading to proptosis and ocular dysfunction. Recent evidence suggests that gut microbiota dysbiosis contributes to GO pathogenesis, particularly through alterations in short-chain fatty acid (SCFA) metabolism. This study investigated the role of Faecalibacterium prausnitzii (F. prausnitzii), a key butyrate-producing bacterium, in modulating GO-related fibrosis and adipogenesis. Clinical fecal sample analysis revealed a significant reduction in F. prausnitzii abundance in GO patients. In a murine GO model, oral administration of F. prausnitzii significantly alleviated periorbital swelling, suppressed collagen deposition, and reduced lipid accumulati

## 13. Midtraining Synthetic Data Generation exploration

Roughly trying to follow the method of [MedReason](https://arxiv.org/abs/2504.00993)

In [94]:
import json
sample_qa = json.load(open("data/gmrepo/taxonomic_profiles/SRR13593011.json"))

In [ ]:
sample_qa

{'run': [{'project_id': 'PRJNA695309',
   'original_sample_description': 'Gut microbiota in Graves disease patients',
   'run_id': 'SRR13593011',
   'experiment_type': 'AMPLICON',
   'instrument_model': 'Illumina MiSeq',
   'nr_reads_sequenced': 224148,
   'host_age': 66.0,
   'sex': 'male',
   'BMI': None,
   'country': 'China',
   'longitude': None,
   'latitude': None,
   'loaded_uid': 1482,
   'QCStatus': 1,
   'QCMessage': 'Forward left trim: 4;Forward right trim: 204;Reverse left trim: 4;Reverse right trim: 201',
   'Original_Project_description': 'To characterize the composition of gut microbiota in GD patients'}],
 'phenotypes_exist': True,
 'phenotypes': [{'disease': 'D006111', 'term': 'Graves Disease'}],
 'genus': [{'loaded_uid': 1482,
   'ncbi_taxon_id': 1730,
   'taxon_rank_level': 'genus',
   'relative_abundance': 66.66666666666666,
   'scientific_name': 'Eubacterium'},
  {'loaded_uid': 1482,
   'ncbi_taxon_id': 216851,
   'taxon_rank_level': 'genus',
   'relative_abundanc

In [ ]:
# load in the kbs:

import pandas as pd

prime_kg = pd.read_csv("data/kb.csv") # this thing is like a gb!

In [100]:
import requests
import time

# organism = "fpr"  # Change to your organism code

# Step 1: Get the organism code for the organism of interest

response = requests.get(f"https://rest.kegg.jp/conv/genome/ncbi-taxid:{sample_qa['genus'][0]['ncbi_taxon_id']}")
# pathway_ids = [line.split("\t")[0].replace("path:", "") 
#                for line in response.text.strip().split("\n")]
print(response.text)

In [107]:
# let's get some genes using the ncbi datasets rest api:
taxid = sample_qa['genus'][0]['ncbi_taxon_id']
# url = f"https://api.ncbi.nlm.nih.gov/datasets/v2/gene/taxon/{taxid}"
url = f"https://api.ncbi.nlm.nih.gov/datasets/v2/taxonomy/taxon/{taxid}/related_ids"
print(url)
response = requests.get(url)


https://api.ncbi.nlm.nih.gov/datasets/v2/taxonomy/taxon/1730/related_ids


In [109]:
len(response.json()['tax_ids'])

20

In [110]:
response.json()['tax_ids']

[1730,
 84372,
 165185,
 84373,
 319438,
 242681,
 242680,
 242682,
 242676,
 242679,
 242678,
 242677,
 207552,
 207551,
 163506,
 163505,
 163552,
 163551,
 163504,
 163591]

In [215]:
# 216851
taxid = sample_qa['genus'][1]['ncbi_taxon_id']
# url = f"https://api.ncbi.nlm.nih.gov/datasets/v2/gene/taxon/{taxid}"
url = f"https://api.ncbi.nlm.nih.gov/datasets/v2/taxonomy/taxon/{taxid}/related_ids"
print(url)
response = requests.get(url)
response.json()

https://api.ncbi.nlm.nih.gov/datasets/v2/taxonomy/taxon/216851/related_ids


{'tax_ids': [216851,
  259314,
  259315,
  2646395,
  3695676,
  2940023,
  2939949,
  2939947,
  2939945,
  2939938,
  2939935,
  2939932,
  2939923,
  2939920,
  2939919,
  2939914,
  2939910,
  2939862,
  2939824,
  2939815],
 'next_page_token': 'EJy3swE='}

In [ ]:


# taxid = sample_qa['genus'][0]['ncbi_taxon_id']
taxid = 81409
url = f"https://api.ncbi.nlm.nih.gov/datasets/v2/genome/taxon/{taxid}"
# url = f"https://api.ncbi.nlm.nih.gov/datasets/v2/taxonomy/taxon/{taxid}/related_ids"
print(url)
response_genes = requests.get(url)

https://api.ncbi.nlm.nih.gov/datasets/v2/genome/taxon/81409


In [120]:
response_genes.json()

{'error': 'Not Found',
 'code': 404,
 'message': 'Your request is invalid. (For more help, see the NCBI Datasets Documentation at https://www.ncbi.nlm.nih.gov/datasets/docs/)'}

In [99]:
sample_qa['genus'][0]['ncbi_taxon_id']

1730

In [122]:
# Let's try the mimedb

mimedb_microbes = pd.read_csv('data/kgs/mimedb_microbes_v2.csv')

In [123]:
mimedb_microbes.shape

(2648, 44)

In [124]:
mimedb_microbes.head()

,id,name,microbe_id,species,kingdom,phylum,ncbi_tax_id,activity,gram,oxygen_requirement,...,export,human_pathogen,genome_ids,parent_id,level,subspecies,serotype,variant,basys2_id,description
0,2,Micromonospora aurantiaca ATCC 27029,MMDBm0000002,Micromonospora aurantiaca,Eubacteria,Actinobacteria,644283.0,Production (export),Positive,Aerobe,...,1,NaN,275,687530.0,NaN,NaN,NaN,NaN,NC_014391.1,Micromonospora aurantiaca ATCC 27029 is a meso...
1,3,Acidaminococcus fermentans DSM 20731,MMDBm0000003,Acidaminococcus fermentans,Eubacteria,Firmicutes,591001.0,NaN,Negative,Anaerobe,...,1,NaN,26,1028.0,NaN,NaN,NaN,NaN,FOWJ00000000.1,Acidaminococcus fermentans DSM 20731 is a meso...
2,4,Acidaminococcus intestini RyC-MR95,MMDBm0000004,Acidaminococcus intestini,Eubacteria,Firmicutes,568816.0,NaN,Negative,Anaerobe,...,1,NaN,36,808970.0,NaN,NaN,NaN,NaN,NC_016077.1,Acidaminococcus intestini RyC-MR95 is an oblig...
3,6,Acinetobacter baumannii,MMDBm0000006,Acinetobacter baumannii,Eubacteria,Proteobacteria,470.0,NaN,Negative,Aerobe,...,1,NaN,1096,NaN,NaN,NaN,NaN,NaN,NC_010410.1,Acinetobacter baumannii is classified as a mes...
4,7,Acinetobacter pittii ANC 4052,MMDBm0000007,Acinetobacter pittii,Eubacteria,Proteobacteria,1217689.0,NaN,Negative,Aerobe,...,1,NaN,2443,572679.0,NaN,NaN,NaN,NaN,LLCW00000000.1,"Acinetobacter pittii ANC 4052 is a mesophile, ..."


In [128]:
mimedb_microbes['metabolism'].head()

0              Polyisoprene rubber degrader
1                                       NaN
2    Asaccharolytic, glutamate is fermented
3                              Prototrophic
4                            Asaccharolytic
Name: metabolism, dtype: str

In [127]:
print("\n".join(mimedb_microbes.columns))

id
name
microbe_id
species
kingdom
phylum
ncbi_tax_id
activity
gram
oxygen_requirement
created_at
updated_at
klass
order
family
genus
strain
metabolism
shape
mobility
flagella_presence
number_of_membranes
optimal_temperature
temperature_range
habitat
biotic_relationship
cell_arrangement
sporulation
energy_source
superkingdom
background
health_type
evidence_type
data_source
export
human_pathogen
genome_ids
parent_id
level
subspecies
serotype
variant
basys2_id
description


In [151]:
mimedb_microbes['biotic_relationship']

0       Free living
1       Free living
2               NaN
3       Free living
4               NaN
           ...     
2643    Free living
2644            NaN
2645            NaN
2646      Symbiotic
2647            NaN
Name: biotic_relationship, Length: 2648, dtype: str

In [129]:
sample_qa['genus']

[{'loaded_uid': 1482,
  'ncbi_taxon_id': 1730,
  'taxon_rank_level': 'genus',
  'relative_abundance': 66.66666666666666,
  'scientific_name': 'Eubacterium'},
 {'loaded_uid': 1482,
  'ncbi_taxon_id': 216851,
  'taxon_rank_level': 'genus',
  'relative_abundance': 33.33333333333333,
  'scientific_name': 'Faecalibacterium'}]

In [135]:
mimedb_microbes.query('genus == "Faecalibacterium"')['description']

121     Faecalibacterium prausnitzii is a Gram-positiv...
122     Faecalibacterium prausnitzii L2-6 is a bacteri...
123     Faecalibacterium prausnitzii SL3/3 is a gram-p...
2456    Faecalibacterium prausnitzii A2-165 is a Gram-...
2457                                                  NaN
2458    Faecalibacterium prausnitzii M21/2 is a Gram-p...
2601    Faecalibacterium faecis is a Gram-positive, ro...
2602    Faecalibacterium intestinale is a Gram-positiv...
2603    Faecalibacterium tardum is a Gram-positive, ro...
Name: description, dtype: str

In [ ]:
mimedb_metabolites = pd.read_csv("data/kgs/mimedb_metabolites_v2.csv")

In [138]:
mimedb_metabolites.head()

,id,name,mime_id,export,description,cas,state,comment,moldb_id,moldb_smiles,...,quantified,predicted,expected,vmh_id,metabolite_type,classification,epa_substance_id,epa_compound_id,microbe_relations,cmmc_inchikey
0,2,Indolylacryloylglycine,MMDBc0057090,1,"Indolylacryloylglycine, also known as IAG or i...",3475-68-1,Expected Solid,This metabolite is not currently associated wi...,NaN,[H]\C(=C(\[H])C1=CNC2=CC=CC=C12)C(O)=NCC(O)=O,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN
1,3,(S)-3-Hydroxyisobutyric acid,MMDBc0057091,1,"Indolylacryloylglycine, also known as IAG or i...",26543-05-5,Expected Solid,NaN,NaN,NaN,...,1.0,NaN,NaN,3hmp,NaN,Hydroxy acids and derivatives,NaN,NaN,416,NaN
2,4,Ethylene glycol,MMDBc0057092,1,NaN,107-21-1,Expected Solid,This is a polymer. Unexported by An Chi on Jul...,NaN,[H]OCCO[H],...,NaN,NaN,NaN,NaN,NaN,Organooxygen compounds,DTXSID8020597,DTXCID40597,9,NaN
3,5,2-Ketobutyric acid,MMDBc0000005,1,"2-Ketobutyric acid, also known as alpha-ketobu...",600-18-0,Expected Solid,NaN,NaN,CCC(=O)C(O)=O,...,1.0,NaN,NaN,2obut,NaN,Keto acids and derivatives,DTXSID9060524,DTXCID5042740,1151,NaN
4,6,Biotin,MMDBc0000006,1,Biotin is an enzyme co-factor present in minut...,58-85-5,Expected Solid,NaN,NaN,[H][C@]12CS[C@@H](CCCCC(O)=O)[C@@]1([H])NC(=O)N2,...,1.0,NaN,NaN,btn,NaN,Biotin and derivatives,NaN,NaN,1425,NaN


In [152]:
mimedb_microbes.head()

,id,name,microbe_id,species,kingdom,phylum,ncbi_tax_id,activity,gram,oxygen_requirement,...,export,human_pathogen,genome_ids,parent_id,level,subspecies,serotype,variant,basys2_id,description
0,2,Micromonospora aurantiaca ATCC 27029,MMDBm0000002,Micromonospora aurantiaca,Eubacteria,Actinobacteria,644283.0,Production (export),Positive,Aerobe,...,1,NaN,275,687530.0,NaN,NaN,NaN,NaN,NC_014391.1,Micromonospora aurantiaca ATCC 27029 is a meso...
1,3,Acidaminococcus fermentans DSM 20731,MMDBm0000003,Acidaminococcus fermentans,Eubacteria,Firmicutes,591001.0,NaN,Negative,Anaerobe,...,1,NaN,26,1028.0,NaN,NaN,NaN,NaN,FOWJ00000000.1,Acidaminococcus fermentans DSM 20731 is a meso...
2,4,Acidaminococcus intestini RyC-MR95,MMDBm0000004,Acidaminococcus intestini,Eubacteria,Firmicutes,568816.0,NaN,Negative,Anaerobe,...,1,NaN,36,808970.0,NaN,NaN,NaN,NaN,NC_016077.1,Acidaminococcus intestini RyC-MR95 is an oblig...
3,6,Acinetobacter baumannii,MMDBm0000006,Acinetobacter baumannii,Eubacteria,Proteobacteria,470.0,NaN,Negative,Aerobe,...,1,NaN,1096,NaN,NaN,NaN,NaN,NaN,NC_010410.1,Acinetobacter baumannii is classified as a mes...
4,7,Acinetobacter pittii ANC 4052,MMDBm0000007,Acinetobacter pittii,Eubacteria,Proteobacteria,1217689.0,NaN,Negative,Aerobe,...,1,NaN,2443,572679.0,NaN,NaN,NaN,NaN,LLCW00000000.1,"Acinetobacter pittii ANC 4052 is a mesophile, ..."


In [139]:
print("\n".join(mimedb_metabolites.columns))

id
name
mime_id
export
description
cas
state
comment
moldb_id
moldb_smiles
moldb_formula
moldb_inchi
moldb_inchikey
moldb_iupac
moldb_logp
moldb_pka
moldb_average_mass
moldb_mono_mass
moldb_alogps_solubility
moldb_alogps_logp
moldb_alogps_logs
moldb_acceptor_count
moldb_donor_count
moldb_rotatable_bond_count
moldb_polar_surface_area
moldb_refractivity
moldb_polarizability
moldb_traditional_iupac
moldb_formal_charge
moldb_physiological_charge
moldb_pka_strongest_acidic
moldb_pka_strongest_basic
created_at
updated_at
hmdb_id
detected
quantified
predicted
expected
vmh_id
metabolite_type
classification
epa_substance_id
epa_compound_id
microbe_relations
cmmc_inchikey


In [148]:
mimedb_metabolites.head()["cas"]

0     3475-68-1
1    26543-05-5
2      107-21-1
3      600-18-0
4       58-85-5
Name: cas, dtype: str

In [ ]:
# Mimedb is not particularly useful...



Ok, version 1 is linking mimedb to hmdb and just taking the diseases from hmdb bc it has a bunch.

In [149]:
# version 1 - find path from Faecalibacterium to Grave's disease via metabolites in mimedb

fb = mimedb_microbes.query('genus == "Faecalibacterium"')




In [150]:
fb

,id,name,microbe_id,species,kingdom,phylum,ncbi_tax_id,activity,gram,oxygen_requirement,...,export,human_pathogen,genome_ids,parent_id,level,subspecies,serotype,variant,basys2_id,description
121,179,Faecalibacterium prausnitzii,MMDBm0000179,Faecalibacterium prausnitzii,Eubacteria,Firmicutes,853.0,NaN,NaN,Anaerobe,...,1,NaN,1278,NaN,NaN,NaN,NaN,NaN,NC_021042.1,Faecalibacterium prausnitzii is a Gram-positiv...
122,180,Faecalibacterium prausnitzii L2-6,MMDBm0000180,Faecalibacterium prausnitzii,Eubacteria,Firmicutes,718252.0,NaN,Structurally positive but stains negative,Anaerobe,...,1,NaN,1692,179.0,NaN,NaN,NaN,NaN,NC_021042.1,Faecalibacterium prausnitzii L2-6 is a bacteri...
123,181,Faecalibacterium prausnitzii SL3/3,MMDBm0000181,Faecalibacterium prausnitzii,Eubacteria,Firmicutes,657322.0,NaN,Structurally positive but stains negative,Anaerobe,...,1,NaN,1674,179.0,NaN,NaN,NaN,NaN,NC_021042.1,Faecalibacterium prausnitzii SL3/3 is a gram-p...
2456,805488,Faecalibacterium prausnitzii A2-165,MMDBm0805488,Faecalibacterium prausnitzii,Eubacteria,Firmicutes,411483.0,NaN,Structurally positive but stains negative,Anaerobe,...,1,NaN,1544,179.0,NaN,NaN,NaN,NaN,NC_021042.1,Faecalibacterium prausnitzii A2-165 is a Gram-...
2457,805489,Faecalibacterium cf. prausnitzii KLE1255,MMDBm0805489,Faecalibacterium prausnitzii,NaN,Firmicutes,748224.0,NaN,Structurally positive but stains negative,Obligate anaerobe,...,1,NaN,NaN,179.0,NaN,NaN,NaN,NaN,NC_021042.1,NaN
2458,805490,Faecalibacterium prausnitzii M21/2,MMDBm0805490,Faecalibacterium prausnitzii,Eubacteria,Firmicutes,411485.0,NaN,Structurally positive but stains negative,Anaerobe,...,1,NaN,1545,179.0,NaN,NaN,NaN,NaN,NC_021042.1,Faecalibacterium prausnitzii M21/2 is a Gram-p...
2601,882206,Faecalibacterium faecis,MMDBm0882194,Faecalibacterium faecis,Bacillati,Bacillota,3133157.0,NaN,Positive,Anaerobic,...,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"Faecalibacterium faecis is a Gram-positive, ro..."
2602,882207,Faecalibacterium intestinale,MMDBm0882195,Faecalibacterium intestinale,Bacillati,Bacillota,3133155.0,NaN,Positive,Anaerobic,...,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Faecalibacterium intestinale is a Gram-positiv...
2603,882208,Faecalibacterium tardum,MMDBm0882196,Faecalibacterium tardum,Bacillati,Bacillota,3133156.0,NaN,Positive,Anaerobic,...,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"Faecalibacterium tardum is a Gram-positive, ro..."


In [153]:
mimedb_bioactivity_microbe = pd.read_csv("data/kgs/bioactivity_microbe.csv")

In [158]:
mimedb_bioactivity_microbe.merge(fb, right_on="microbe_id", left_on="Microbe ID", how="inner")["Related health outcome"].unique()

<ArrowStringArray>
['Chronic kidney disease;Atherosclerosis;Autism;Autism spectrum disorders;Celiac disease, coaliac disease;Cholestasis;Cirrhosis;Colorectal cancer;Colon polyps;Crohn's disease;Inactive crohn's disease;Cystic fibrosis;Type 1 diabetes;Type 2 diabetes;Fatigue;Fibromyalgia;Heart failure;Hepatitis b;Inflammatory bowel disease;Keratitis;Lung cancer;Metabolic syndrome;Migraine;Multiple sclerosis;Idiopathic nephrotic syndrome;Obesity;Pancreatitis;Parkinson's disease;Ulcerative colitis;Non-alcoholic fatty liver disease;Clostridium difficile infection;Pervasive developmental disorder nos;Hyperlipidemia;Systemic lupus erythematosus;Myasthenia gravis']
Length: 1, dtype: str

In [159]:
diseases = mimedb_bioactivity_microbe.merge(fb, right_on="microbe_id", left_on="Microbe ID", how="inner")["Related health outcome"].unique()[0]

In [162]:
"grave" in diseases.lower()

False

I guess it's kind of sad that Grave's disease isn't mentioned here... That makes the task much more difficult...
Interestinly, there are a ton of other auto-immune diseases mentioned.

In [170]:
all_diseases = []
for disease in mimedb_bioactivity_microbe["Related health outcome"]:
    all_diseases.extend(map( str.lower, disease.split(";")))
all_diseases = sorted(list(set(all_diseases)))

In [172]:
any(["grave" in disease for disease in all_diseases])

False

So... there might be a connection, but it's not in this database... Ok... so this one should probably

In [173]:
len(all_diseases)

204

In [175]:
# interesting in gmrepo, I think we have something like ~221 unique diseases

# gm_repo_runs = pd.read_excel("data/gmrepo/all_runs_metadata.xlsx")
gmrepo_diseases = []
import glob
for fn in glob.glob("data/gmrepo/diseases/*.json"):
    with open(fn) as f:
        gmrepo_diseases.append(json.load(f))

In [182]:
gmrepo_disease_labels = []
for d in gmrepo_diseases:
    try:
        gmrepo_disease_labels.extend([t['label'].lower() for t in d['terms']])
    except KeyError:
        print(f"KeyError for disease {d.get('disease_id', 'unknown')}, skipping.")
        continue

KeyError for disease unknown, skipping.


In [184]:
len(set(gmrepo_disease_labels).intersection(set(all_diseases)))

52

So.. there are 52 diseases in common between MiMeDB and GMRepo - that's not great... considering that both have over 200. 

In [185]:
set(all_diseases).difference(set(gmrepo_disease_labels))

{' empyema',
 'abdominal cramps',
 'abdominal pain',
 'abscess',
 'acquired immunodeficiency syndrome',
 'actinomycosis',
 'acute cholecystitis',
 'acute diarrhea',
 'acute gastroenteritis',
 'acute otitis',
 'allergy',
 'ards',
 'arsenic exposure',
 'atherosclerosis',
 'atrophic gastritis',
 'autism',
 'bacterial vaginosis',
 'bartholin abscess',
 'bartholin cyst',
 'bladder cancer',
 'blister',
 'bronchitis',
 'caries',
 'carious lesions',
 'celiac disease, coaliac disease',
 'chest infection',
 'cholecystitis',
 'cholera infection',
 'chronic kidney disease',
 'chronic lung disease',
 'chronic obstructive pulmonary disease',
 'chronic rhinitis',
 'chronic sinusitis',
 'chronic ulcer',
 'cirrhosis',
 'ckd',
 'coeliac disease',
 'colitis',
 'colon polyps',
 'copd exacerbation',
 'coronary artery disease',
 'cough',
 'cutaneous lesion',
 'cystitis',
 'decubital ulcer',
 'dental caries',
 'diarrheae',
 'diphtheria',
 'diverticular disease',
 'down syndrome',
 'duodenal ulcer',
 'duodeni

In [187]:
sorted(list(set(gmrepo_disease_labels).difference(set(all_diseases))))[:100]

['2-d gel electrophoresis',
 '2-d polyacrylamide gel electrophoresis',
 '2019 novel coronavirus disease',
 '2019 novel coronavirus infection',
 '2019-ncov disease',
 '2019-ncov infection',
 '2d gel electrophoresis',
 '2d page',
 '2d polyacrylamide gel electrophoresis',
 'abscess, amebic',
 'acanthamebiasis',
 'acanthamoeba infection',
 'acne inversa',
 'acne vulgaris',
 'acquisition, health facility',
 'acromegaly due to pituitary adenoma',
 'acute confusional senile dementia',
 'acute edematous pancreatitis',
 'acute hypercapnic respiratory failure',
 'acute hypoxemic respiratory failure',
 'acute ischemic stroke',
 'acute myelogenous leukemia',
 'acute myeloid leukemia',
 'acute myeloid leukemia with maturation',
 'acute myeloid leukemia without maturation',
 'acute pancreatitis',
 'acute relapsing multiple sclerosis',
 'adamantiades-behcet disease',
 'addh',
 'adenocarcinoma of kidney',
 'adenocarcinoma, renal',
 'adenocarcinoma, renal cell',
 'adenomatous polyposis coli',
 'adenoma

In [179]:
d

{'error': {'unknown': [['Illegal character in path at index 7: D015228|D010195']]}}

Version 2: Use gutMGene as an intermediary

In [189]:
microbe_host_gene = pd.read_csv("data/kgs/gutMGene/Gut Microbe-Host Gene.csv", encoding='GB18030')

In [203]:
microbe_host_gene["Gut Microbiota NCBI ID"].unique().sort()

In [204]:
microbe_host_gene.query("`Gut Microbiota NCBI ID` == 1730")

,Index,PMID,Gut Microbiota,Gut Microbiota NCBI ID,Rank,Strain,Gene,Gene ID,Alteration,high/low-throughput,Associative mode,human/mouse,Sample,Experimental method,Measurement?technique,Description,Condition,DOID


In [212]:
# 'ncbi_taxon_id': 216851,
#    'taxon_rank_level': 'genus',
#    'relative_abundance': 33.33333333333333,
#    'scientific_name': 'Faecalibacterium'}]


for row in microbe_host_gene.itertuples():
    # print(row)
    if "Faecalibacterium" in row._3:
        print(f"Got one:", row._1, row._4, row._3)
        # print(microbe_host_gene.query("`Gut Microbiota NCBI ID` == @row._1"))
    # if row._1 in [1730, 84372, 165185, 84373, 163552, 163551, 163504, 163591]:
    #     print(f"Taxa ID: {row._1}")
    #     print(microbe_host_gene.query("`Gut Microbiota NCBI ID` == @row._1"))

Got one: 5 853.0 Faecalibacterium prausnitzii
Got one: 5 853.0 Faecalibacterium prausnitzii
Got one: 5 853.0 Faecalibacterium prausnitzii
Got one: 11 853.0 Faecalibacterium prausnitzii
Got one: 11 853.0 Faecalibacterium prausnitzii
Got one: 11 853.0 Faecalibacterium prausnitzii
Got one: 11 853.0 Faecalibacterium prausnitzii
Got one: 14 853.0 Faecalibacterium prausnitzii
Got one: 14 853.0 Faecalibacterium prausnitzii
Got one: 14 853.0 Faecalibacterium prausnitzii
Got one: 28 853.0 Faecalibacterium prausnitzii
Got one: 28 853.0 Faecalibacterium prausnitzii
Got one: 30 853.0 Faecalibacterium prausnitzii
Got one: 30 853.0 Faecalibacterium prausnitzii
Got one: 30 853.0 Faecalibacterium prausnitzii
Got one: 30 853.0 Faecalibacterium prausnitzii
Got one: 30 853.0 Faecalibacterium prausnitzii
Got one: 40 853.0 Faecalibacterium prausnitzii
Got one: 40 853.0 Faecalibacterium prausnitzii
Got one: 40 853.0 Faecalibacterium prausnitzii
Got one: 40 853.0 Faecalibacterium prausnitzii
Got one: 42 853.

In [205]:
for taxa_id in [1730,
 84372,
 165185,
 84373,
 319438,
 242681,
 242680,
 242682,
 242676,
 242679,
 242678,
 242677,
 207552,
 207551,
 163506,
 163505,
 163552,
 163551,
 163504,
 163591]:
    print(f"Taxa ID: {taxa_id}")
    print(microbe_host_gene.query("`Gut Microbiota NCBI ID` == @taxa_id"))

Taxa ID: 1730
Empty DataFrame
Columns: [Index, PMID, Gut Microbiota, Gut Microbiota NCBI ID, Rank, Strain, Gene, Gene ID, Alteration, high/low-throughput, Associative mode, human/mouse, Sample, Experimental method, Measurement?technique, Description, Condition, DOID]
Index: []
Taxa ID: 84372
Empty DataFrame
Columns: [Index, PMID, Gut Microbiota, Gut Microbiota NCBI ID, Rank, Strain, Gene, Gene ID, Alteration, high/low-throughput, Associative mode, human/mouse, Sample, Experimental method, Measurement?technique, Description, Condition, DOID]
Index: []
Taxa ID: 165185
Empty DataFrame
Columns: [Index, PMID, Gut Microbiota, Gut Microbiota NCBI ID, Rank, Strain, Gene, Gene ID, Alteration, high/low-throughput, Associative mode, human/mouse, Sample, Experimental method, Measurement?technique, Description, Condition, DOID]
Index: []
Taxa ID: 84373
Empty DataFrame
Columns: [Index, PMID, Gut Microbiota, Gut Microbiota NCBI ID, Rank, Strain, Gene, Gene ID, Alteration, high/low-throughput, Associa

In [193]:
fb

,id,name,microbe_id,species,kingdom,phylum,ncbi_tax_id,activity,gram,oxygen_requirement,...,export,human_pathogen,genome_ids,parent_id,level,subspecies,serotype,variant,basys2_id,description
121,179,Faecalibacterium prausnitzii,MMDBm0000179,Faecalibacterium prausnitzii,Eubacteria,Firmicutes,853.0,NaN,NaN,Anaerobe,...,1,NaN,1278,NaN,NaN,NaN,NaN,NaN,NC_021042.1,Faecalibacterium prausnitzii is a Gram-positiv...
122,180,Faecalibacterium prausnitzii L2-6,MMDBm0000180,Faecalibacterium prausnitzii,Eubacteria,Firmicutes,718252.0,NaN,Structurally positive but stains negative,Anaerobe,...,1,NaN,1692,179.0,NaN,NaN,NaN,NaN,NC_021042.1,Faecalibacterium prausnitzii L2-6 is a bacteri...
123,181,Faecalibacterium prausnitzii SL3/3,MMDBm0000181,Faecalibacterium prausnitzii,Eubacteria,Firmicutes,657322.0,NaN,Structurally positive but stains negative,Anaerobe,...,1,NaN,1674,179.0,NaN,NaN,NaN,NaN,NC_021042.1,Faecalibacterium prausnitzii SL3/3 is a gram-p...
2456,805488,Faecalibacterium prausnitzii A2-165,MMDBm0805488,Faecalibacterium prausnitzii,Eubacteria,Firmicutes,411483.0,NaN,Structurally positive but stains negative,Anaerobe,...,1,NaN,1544,179.0,NaN,NaN,NaN,NaN,NC_021042.1,Faecalibacterium prausnitzii A2-165 is a Gram-...
2457,805489,Faecalibacterium cf. prausnitzii KLE1255,MMDBm0805489,Faecalibacterium prausnitzii,NaN,Firmicutes,748224.0,NaN,Structurally positive but stains negative,Obligate anaerobe,...,1,NaN,NaN,179.0,NaN,NaN,NaN,NaN,NC_021042.1,NaN
2458,805490,Faecalibacterium prausnitzii M21/2,MMDBm0805490,Faecalibacterium prausnitzii,Eubacteria,Firmicutes,411485.0,NaN,Structurally positive but stains negative,Anaerobe,...,1,NaN,1545,179.0,NaN,NaN,NaN,NaN,NC_021042.1,Faecalibacterium prausnitzii M21/2 is a Gram-p...
2601,882206,Faecalibacterium faecis,MMDBm0882194,Faecalibacterium faecis,Bacillati,Bacillota,3133157.0,NaN,Positive,Anaerobic,...,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"Faecalibacterium faecis is a Gram-positive, ro..."
2602,882207,Faecalibacterium intestinale,MMDBm0882195,Faecalibacterium intestinale,Bacillati,Bacillota,3133155.0,NaN,Positive,Anaerobic,...,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Faecalibacterium intestinale is a Gram-positiv...
2603,882208,Faecalibacterium tardum,MMDBm0882196,Faecalibacterium tardum,Bacillati,Bacillota,3133156.0,NaN,Positive,Anaerobic,...,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"Faecalibacterium tardum is a Gram-positive, ro..."


## 12. Midtraining Dataset Exploration
6/23/2026

### 12.1 MGnify

In [ ]:
from datasets import load_dataset
from pathlib import Path

In [16]:
token = (Path.home() / ".hf_token_fs").open().read().strip()
mgnify = load_dataset("microbiome-fm/mgnify-microbiome", token=token)

README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/90 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/90 [00:00<?, ?it/s]

data/train-00000-of-00090.parquet:   0%|          | 0.00/23.9M [00:00<?, ?B/s]

data/train-00001-of-00090.parquet:   0%|          | 0.00/25.3M [00:00<?, ?B/s]

...

data/train-00087-of-00090.parquet:   0%|          | 0.00/23.6M [00:00<?, ?B/s]

data/train-00088-of-00090.parquet:   0%|          | 0.00/22.7M [00:00<?, ?B/s]

data/train-00089-of-00090.parquet:   0%|          | 0.00/25.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/273485 [00:00<?, ? examples/s]

Loading dataset shards:   0%|          | 0/81 [00:03<?, ?it/s]

In [18]:
mgnify['train'][0]

{'contigs': {'data': [],
  'links': {'next': None, 'prev': None},
  'meta': {'pagination': {'count': 0}}},
 'taxonomy': {'data': [],
  'links': {'first': 'https://www.ebi.ac.uk/metagenomics/api/v1/analyses/MGYA00657896/taxonomy?page=1',
   'last': 'https://www.ebi.ac.uk/metagenomics/api/v1/analyses/MGYA00657896/taxonomy?page=1',
   'next': None,
   'prev': None},
  'meta': {'pagination': {'count': 0, 'page': 1, 'pages': 1}}},
 'taxonomy_itsonedb': {'data': [],
  'links': {'first': 'https://www.ebi.ac.uk/metagenomics/api/v1/analyses/MGYA00657896/taxonomy/itsonedb?page=1',
   'last': 'https://www.ebi.ac.uk/metagenomics/api/v1/analyses/MGYA00657896/taxonomy/itsonedb?page=1',
   'next': None,
   'prev': None},
  'meta': {'pagination': {'count': 0, 'page': 1, 'pages': 1}}},
 'taxonomy_lsu': {'data': [],
  'links': {'first': 'https://www.ebi.ac.uk/metagenomics/api/v1/analyses/MGYA00657896/taxonomy/lsu?page=1',
   'last': 'https://www.ebi.ac.uk/metagenomics/api/v1/analyses/MGYA00657896/taxono

In [19]:
mgnify

DatasetDict({
    train: Dataset({
        features: ['contigs', 'taxonomy', 'taxonomy_itsonedb', 'taxonomy_lsu', 'taxonomy_ssu', 'taxonomy_unite', 'go_slim', 'antismash_gene_clusters', 'go_terms', 'genome_properties', 'interpro_identifiers', 'kegg_orthologs', 'pfam_entries', 'sample_id', 'run_id', 'analysis_id', 'assembly_id', 'analysis_attributes', 'analysis_relationships', 'sample_metadata'],
        num_rows: 273485
    })
})

In [25]:
for i in range(200_000):
    if mgnify["train"][i]['contigs']['data']:
        print(i)
        break

29


In [26]:
mgnify["train"][i]['contigs']

{'data': [{'attributes': {'accession': 'MGYA00652274',
    'analysis-id': '652274',
    'contig-id': 'ERZ19591242.1-NODE-9-length-583-cov-4.0000',
    'coverage': 4.0,
    'has-antismash': False,
    'has-cog': True,
    'has-go': True,
    'has-interpro': True,
    'has-kegg': True,
    'has-kegg-module': False,
    'has-pfam': True,
    'job-id': 652274,
    'length': 583,
    'pipeline-version': '5.0'},
   'id': 'ERZ19591242.1-NODE-9-length-583-cov-4.0000',
   'type': 'analysis-job-contigs'},
  {'attributes': {'accession': 'MGYA00652274',
    'analysis-id': '652274',
    'contig-id': 'ERZ19591242.2-NODE-11-length-612-cov-4.0000',
    'coverage': 4.0,
    'has-antismash': False,
    'has-cog': True,
    'has-go': True,
    'has-interpro': True,
    'has-kegg': True,
    'has-kegg-module': False,
    'has-pfam': True,
    'job-id': 652274,
    'length': 612,
    'pipeline-version': '5.0'},
   'id': 'ERZ19591242.2-NODE-11-length-612-cov-4.0000',
   'type': 'analysis-job-contigs'},
  {'

### 12.2 UHGG


In [27]:
import pandas as pd
uhgg_10 = pd.read_csv("data/uhgg/uhgg_genomes_all_metadata_v2.0.2_first10.csv")

In [28]:
uhgg_10

,Genome,Genome_type,Length,N_contigs,N50,GC_content,Completeness,Contamination,rRNA_5S,rRNA_16S,rRNA_23S,tRNAs,Genome_accession,Species_rep,Lineage,Sample_accession,Study_accession,Country,Continent,FTP_download
0,MGYG000000001,Isolate,3221717,129,47870,28.24,98.59,0.70,88.24,99.74,99.83,20,GUT_GENOME000001,MGYG000000001,d__Bacteria;p__Firmicutes_A;c__Clostridia;o__P...,ERS370061,ERP105624,United Kingdom,Europe,ftp://ftp.ebi.ac.uk/pub/databases/metagenomics...
1,MGYG000000002,Isolate,4441003,71,210803,42.53,99.37,0.00,82.35,95.37,86.19,19,GUT_GENOME000004,MGYG000000002,d__Bacteria;p__Firmicutes_A;c__Clostridia;o__L...,ERS370064,ERP105624,United Kingdom,Europe,ftp://ftp.ebi.ac.uk/pub/databases/metagenomics...
2,MGYG000000003,Isolate,3234232,18,557292,58.43,100.00,0.00,94.96,99.67,99.66,20,GUT_GENOME000008,MGYG000000003,d__Bacteria;p__Bacteroidota;c__Bacteroidia;o__...,ERS370070,ERP105624,United Kingdom,Europe,ftp://ftp.ebi.ac.uk/pub/databases/metagenomics...
3,MGYG000000004,Isolate,3707250,71,173568,54.07,98.66,0.22,93.28,99.67,99.83,20,GUT_GENOME000010,MGYG000000004,d__Bacteria;p__Firmicutes_A;c__Clostridia;o__O...,ERS370072,ERP105624,United Kingdom,Europe,ftp://ftp.ebi.ac.uk/pub/databases/metagenomics...
4,MGYG000000005,Isolate,3932530,22,584818,28.57,99.30,0.00,89.08,99.67,99.83,20,GUT_GENOME000017,MGYG000000005,d__Bacteria;p__Firmicutes_A;c__Clostridia;o__P...,ERS417217,ERP105624,United Kingdom,Europe,ftp://ftp.ebi.ac.uk/pub/databases/metagenomics...
5,MGYG000000006,Isolate,2823639,31,121380,32.64,99.26,1.39,94.96,99.74,99.76,20,GUT_GENOME000020,MGYG000000006,d__Bacteria;p__Firmicutes;c__Bacilli;o__Staphy...,ERS417220,ERP105624,United Kingdom,Europe,ftp://ftp.ebi.ac.uk/pub/databases/metagenomics...
6,MGYG000000007,Isolate,2017471,17,266288,35.49,98.94,0.00,94.12,99.67,99.83,20,GUT_GENOME000021,MGYG000000007,d__Bacteria;p__Firmicutes;c__Bacilli;o__Lactob...,ERS417221,ERP105624,United Kingdom,Europe,ftp://ftp.ebi.ac.uk/pub/databases/metagenomics...
7,MGYG000000008,Isolate,1972012,39,93501,34.34,99.22,0.78,94.96,99.67,99.76,20,GUT_GENOME000022,MGYG000000008,d__Bacteria;p__Firmicutes;c__Bacilli;o__Lactob...,ERS417222,ERP105624,United Kingdom,Europe,ftp://ftp.ebi.ac.uk/pub/databases/metagenomics...
8,MGYG000000009,Isolate,2221226,144,33382,39.77,99.21,0.00,94.12,80.82,98.29,20,GUT_GENOME000024,MGYG000000009,d__Bacteria;p__Firmicutes;c__Bacilli;o__Lactob...,ERS417224,ERP105624,United Kingdom,Europe,ftp://ftp.ebi.ac.uk/pub/databases/metagenomics...


In [67]:
uhgg_all = pd.read_csv("data/uhgg/uhgg_genomes_all_metadata_v2.0.2.csv")
uhgg_all.shape

(289232, 20)

In [69]:
uhgg_all["Genome"][-10:]

289222    MGYG000289384
289223    MGYG000289385
289224    MGYG000289386
289225    MGYG000289387
289226    MGYG000289388
289227    MGYG000289389
289228    MGYG000289390
289229    MGYG000289391
289230    MGYG000289392
289231    MGYG000289393
Name: Genome, dtype: str

### 12.3 GMRepo derisking

I think it makes sense to use GMRepo to derisk - we're going to try to generate the kind of reasoning traces we might expect a model to use to do disease classification.

In [61]:
# PRJDB4064

import pandas as pd

gm_repo_all_projs = pd.read_excel("data/gmrepo/all_projects_metadata.xlsx")

In [91]:
gm_repo_all_projs.shape

(937, 15)

In [34]:
gm_repo_all_projs.head()

,id,checking,Project_ID,First_Project_ID,Second_Project_ID,Original_Project_description,Curated_Project_description,has_it_been_collected,why_not_collect,Number_of_samples,Number_of_runs,Population,Related_Publications,isIndepthAnalzed,PMCID
0,1,0.0,PRJDB3601,PRJDB3601,DRP003048,The human gut metagenome Third Party Annotatio...,NaN,N,part phenotype data,255,400,Japan,"29874232, 39122003, 35240043, 26951067, 369848...",0.0,"PMC6005635, PMC11418011, PMC9096808, PMC483342..."
1,2,NaN,PRJDB10485,PRJDB10485,DRP007092,This project is to analyze the dysbiosis of fe...,NaN,Y,NaN,188,188,NaN,NaN,NaN,NaN
2,3,NaN,PRJDB10526,PRJDB10526,DRP007218,16S rDNA amplicon sequence of gut microbiota o...,NaN,Y,NaN,297,297,NaN,NaN,NaN,NaN
3,4,NaN,PRJDB10528,PRJDB10528,DRP007219,16S rDNA amplicon sequence of gut microbiota o...,NaN,Y,NaN,387,387,NaN,NaN,NaN,"PMC9388534,\nPMC8139087"
4,5,NaN,PRJDB10529,PRJDB10529,DRP007222,16S rDNA amplicon sequence of gut microbiota o...,NaN,Y,NaN,175,175,NaN,NaN,NaN,"PMC8139087,\nPMC9697123"


In [35]:
gm_repo_all_projs.query("Project_ID == 'PRJDB4064'")

,id,checking,Project_ID,First_Project_ID,Second_Project_ID,Original_Project_description,Curated_Project_description,has_it_been_collected,why_not_collect,Number_of_samples,Number_of_runs,Population,Related_Publications,isIndepthAnalzed,PMCID
496,497,NaN,PRJDB4064,PRJDB4064,DRP003174,microbiota of the feces obtained from 16 Japan...,NaN,Y,NaN,56,56,NaN,NaN,NaN,NaN


In [37]:
import glob
import json


taxonomies_fps = glob.glob("data/gmrepo/taxonomic_profiles/*.json")
taxonomies = []
for fp in taxonomies_fps:
    with open(fp) as f:
        taxonomies.append(json.load(f))

In [45]:
unique_proj_ids_df = pd.DataFrame({"Project_ID": ([t['run'][0]['project_id'] for t in taxonomies])}).drop_duplicates()

In [49]:
unique_proj_ids_df["Project_ID"].dtype

<StringDtype(na_value=nan)>

In [50]:
gm_repo_all_projs["Project_ID"].dtype

<StringDtype(na_value=nan)>

In [52]:
gm_repo_all_projs[gm_repo_all_projs["Project_ID"].isin(unique_proj_ids_df["Project_ID"])] #.join(unique_proj_ids_df, how="inner", on="Project_ID").head()

,id,checking,Project_ID,First_Project_ID,Second_Project_ID,Original_Project_description,Curated_Project_description,has_it_been_collected,why_not_collect,Number_of_samples,Number_of_runs,Population,Related_Publications,isIndepthAnalzed,PMCID
6,7,NaN,PRJDB11949,PRJDB11949,DRP007553,A project to clarify the intestinal microbiota...,NaN,Y,NaN,169,169,NaN,NaN,NaN,NaN
10,11,NaN,PRJEB23348,PRJEB23348,ERP105097,ADEM-study,NaN,Y,NaN,237,237,NaN,NaN,NaN,NaN
11,12,NaN,PRJEB28700,PRJEB28700,ERP110943,A comprehensive analysis of the faecal microbi...,NaN,Y,NaN,56,56,NaN,32008578,NaN,PMC6996195
13,14,NaN,PRJEB35990,PRJEB35990,ERP119118,Data generated from 16S rRNA sequencing of tum...,NaN,Y,NaN,508,508,NaN,34638284,NaN,PMC8507728
16,17,NaN,PRJEB50447,PRJEB50447,ERP135028,Although some human studies have reported gut ...,NaN,Y,NaN,78,78,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
922,923,NaN,PRJNA836534,PRJNA836534,NaN,We found alterd intestinal microbes in sclerit...,NaN,Y,NaN,31,31,NaN,35756002,NaN,PMC9218904
923,924,NaN,PRJEB67303,PRJEB67303,NaN,Stool and oral microbiome in chronic lymphocyt...,NaN,Y,NaN,197,197,NaN,39328278,NaN,PMC11425030
926,927,NaN,PRJNA586753,PRJNA586753,NaN,"16s rRNA gene sequencing on pulmonar, oral, an...",NaN,Y,NaN,125,125,NaN,NaN,NaN,NaN
929,930,NaN,PRJEB50376,PRJEB50376,NaN,The aim of this study is to assess the link be...,NaN,Y,NaN,118,118,NaN,NaN,NaN,NaN


In [54]:
gm_repo_all_projs[gm_repo_all_projs["Project_ID"].isin(unique_proj_ids_df["Project_ID"]) & (~gm_repo_all_projs["PMCID"].isna())]

,id,checking,Project_ID,First_Project_ID,Second_Project_ID,Original_Project_description,Curated_Project_description,has_it_been_collected,why_not_collect,Number_of_samples,Number_of_runs,Population,Related_Publications,isIndepthAnalzed,PMCID
11,12,NaN,PRJEB28700,PRJEB28700,ERP110943,A comprehensive analysis of the faecal microbi...,NaN,Y,NaN,56,56,NaN,32008578,NaN,PMC6996195
13,14,NaN,PRJEB35990,PRJEB35990,ERP119118,Data generated from 16S rRNA sequencing of tum...,NaN,Y,NaN,508,508,NaN,34638284,NaN,PMC8507728
17,18,NaN,PRJEB75181,PRJEB75181,ERP159768,Environmental Enteric Dysfunction (EED) is a s...,NaN,Y,NaN,43,43,NaN,38742887,NaN,PMC11332341
22,23,NaN,PRJNA1070593,PRJNA1070593,SRP486424,Achieving sustained virologic response (SVR) i...,NaN,Y,NaN,192,192,NaN,38650735,NaN,PMC11033736
36,37,NaN,PRJNA565548,PRJNA565548,SRP221652,Nasopharyngeal carcinoma (NPC) is a malignant ...,NaN,Y,NaN,70,69,NaN,31867274,NaN,PMC6908496
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
920,921,NaN,PRJEB52971,PRJEB52971,NaN,This study analyzed the gut microbiota composi...,NaN,Y,NaN,124,124,NaN,35983031,NaN,PMC9378784
921,922,NaN,PRJNA904086,PRJNA904086,NaN,SLE feces and vagina microbiota,NaN,Y,NaN,120,120,NaN,"36969178,39364592",NaN,"PMC10036376,PMC11450379"
922,923,NaN,PRJNA836534,PRJNA836534,NaN,We found alterd intestinal microbes in sclerit...,NaN,Y,NaN,31,31,NaN,35756002,NaN,PMC9218904
923,924,NaN,PRJEB67303,PRJEB67303,NaN,Stool and oral microbiome in chronic lymphocyt...,NaN,Y,NaN,197,197,NaN,39328278,NaN,PMC11425030


In [71]:
unique_proj_ids_df = pd.DataFrame({
    "Project_ID": ([t['run'][0]['project_id'] for t in taxonomies]),
    "run_id": ([t['run'][0]['run_id'] for t in taxonomies]),
    "disease": ([t['phenotypes'][0]['term'] for t in taxonomies]),
}).drop_duplicates()

In [56]:
unique_proj_ids_df.head()

,Project_ID,disease
0,PRJNA978061,Psychotic Disorders
1,PRJNA898338,Sotos Syndrome
2,PRJNA799832,Pituitary Neoplasms
3,PRJNA669563,Prader-Willi Syndrome
4,PRJNA799190,Helminthiasis


In [72]:
unique_proj_ids_df.shape

(2120, 3)

In [63]:
# gm_repo_all_projs["Project_ID"] = gm_repo_all_projs["Project_ID"].astype(str)
# unique_proj_ids_df["Project_ID"] = unique_proj_ids_df["Project_ID"].astype(str)
gm_repo_all_projs_sub = gm_repo_all_projs.merge(unique_proj_ids_df, how="inner", on="Project_ID")

In [66]:
gm_repo_all_projs_sub.head(20)

,id,checking,Project_ID,First_Project_ID,Second_Project_ID,Original_Project_description,Curated_Project_description,has_it_been_collected,why_not_collect,Number_of_samples,Number_of_runs,Population,Related_Publications,isIndepthAnalzed,PMCID,disease
0,7,NaN,PRJDB11949,PRJDB11949,DRP007553,A project to clarify the intestinal microbiota...,NaN,Y,NaN,169,169,NaN,NaN,NaN,NaN,HIV Infections
1,11,NaN,PRJEB23348,PRJEB23348,ERP105097,ADEM-study,NaN,Y,NaN,237,237,NaN,NaN,NaN,NaN,Asthma
2,12,NaN,PRJEB28700,PRJEB28700,ERP110943,A comprehensive analysis of the faecal microbi...,NaN,Y,NaN,56,56,NaN,32008578,NaN,PMC6996195,Strongyloides stercoralis
3,14,NaN,PRJEB35990,PRJEB35990,ERP119118,Data generated from 16S rRNA sequencing of tum...,NaN,Y,NaN,508,508,NaN,34638284,NaN,PMC8507728,Colorectal Neoplasms
4,17,NaN,PRJEB50447,PRJEB50447,ERP135028,Although some human studies have reported gut ...,NaN,Y,NaN,78,78,NaN,NaN,NaN,NaN,Alzheimer Disease
5,18,NaN,PRJEB75181,PRJEB75181,ERP159768,Environmental Enteric Dysfunction (EED) is a s...,NaN,Y,NaN,43,43,NaN,38742887,NaN,PMC11332341,Malnutrition
6,19,NaN,PRJNA1038771,PRJNA1038771,SRP471701,To explore the gut microbial characteristics i...,NaN,Y,NaN,184,184,NaN,NaN,NaN,NaN,Hepatolenticular Degeneration
7,23,NaN,PRJNA1070593,PRJNA1070593,SRP486424,Achieving sustained virologic response (SVR) i...,NaN,Y,NaN,192,192,NaN,38650735,NaN,PMC11033736,Hepatitis C
8,26,NaN,PRJNA1125274,PRJNA1125274,SRP514616,Analysis of the gut microbiota in patients wit...,NaN,Y,NaN,289,289,NaN,NaN,NaN,NaN,Sepsis
9,28,NaN,PRJNA342486,PRJNA342486,SRP087829,Fecal metabolomics in pediatric spondyloarthri...,NaN,Y,NaN,41,41,NaN,NaN,NaN,NaN,Spondylarthritis


In [65]:
gm_repo_all_projs_sub.shape

(406, 16)

In [75]:
unique_proj_ids_df.query("Project_ID == 'PRJNA695309'")

,Project_ID,run_id,disease
452,PRJNA695309,SRR13593011,Graves Disease
670,PRJNA695309,SRR13593003,Graves Disease
702,PRJNA695309,SRR13593019,Graves Disease
823,PRJNA695309,SRR13593044,Graves Disease
929,PRJNA695309,SRR13592997,Graves Disease
939,PRJNA695309,SRR13593023,Graves Disease
1172,PRJNA695309,SRR13593034,Graves Disease
1379,PRJNA695309,SRR13593021,Graves Disease
1780,PRJNA695309,SRR13593015,Graves Disease
2081,PRJNA695309,SRR13593036,Graves Disease


Ok, let's use Graves diseae as an example:
https://pmc.ncbi.nlm.nih.gov/articles/PMC8132172/

filepath: gmrepo/taxonomic_profiles/SRR13593011.json

The prompt looks like:

```markdown
Given the following taxonomic profile of the abundance of certain bacteria in a human gut sample, predict what disease the human has or say "Healthy" if there is no disease.

Taxonomic Profile (genus level):
Eubacterium: 66.667%
Faecalibacterium: 33.333%
```

Which is kind of sad for an example... but maybe it's ok...


Relevant part of paper:
> Using 16S rRNA gene amplification and sequencing, the overall bacterial richness and diversity were found to be similar between GD patients and healthy controls. However, principal coordinate analysis and partial least squares-discriminant analysis showed that the overall gut microbiota composition was significantly different (ANOSIM; p < 0.001). The linear discriminant analysis effect size revealed that Firmicutes phylum decreased in GD patients, with a corresponding increase in Bacteroidetes phylum compared to healthy controls. In addition, the families Prevotellaceae, and Veillonellaceae and the genus Prevotella_9 were closely associated with GD patients, while the families Lachnospiraceae and Ruminococcaceae and the **genera Faecalibacterium**, Lachnospira, and Lachnospiraceae NK4A136 were associated with healthy controls.


So an example trace could look like:
```
This patient has relatively small amount of Faecalibacterium relative to healthy controls, which suggests that the patient might have Graves disease.
```

Sure... but like... does this make sense?


> A previous research showed that Faecalibacterium was a protective factor, owing to its negative correlation with thyroid stimulating immunoglobulin antibody related to GD patients 



Let's seee... so from mimedb:
- https://mimedb.org/microbes/MMDBm0882194
- https://mimedb.org/microbes/MMDBm0882195
- https://mimedb.org/microbes/MMDBm0882196
A bunch of different Faecalibacterium. E.g.:

> Faecalibacterium faecis is a Gram-positive, rod-shaped anaerobic bacterium predominantly found in the human gut. This microbe thrives in the complex ecosystem of the gastrointestinal tract, where it plays a pivotal role in the fermentation of dietary fibers, particularly starch. During this metabolic process, F. faecis produces short-chain fatty acids, notably butyrate, which is critical for colonic health, as it serves as a primary energy source for colonic epithelial cells. Additionally, it synthesizes essential vitamins such as riboflavin and folate, alongside amino acids like glutamate, contributing to the overall nutritional profile of the gut microbiome. The anaerobic nature of F. faecis means it thrives in environments lacking oxygen, which is typical for the anaerobic sections of the human intestines. This specialization enables it to effectively compete with other gut microbiota, contributing to the balance of this microbial community. The presence of F. faecis is often associated with a healthy gut microbiome; its abundance can reflect a diet rich in fibers, which promote its growth and metabolic functions.Research indicates that butyrate, produced by F. faecis, has anti-inflammatory properties and may protect against various gastrointestinal disorders, including inflammatory bowel disease. Moreover, the microbe's involvement in the fermentation process can help regulate gut motility and maintain a healthy gut barrier, thus playing a crucial role in metabolic health and disease prevention. Its interactions within the gut microbiome are a focus of ongoing research, highlighting its potential significance in understanding gut health and disease dynamics.

keywords: "butyrate", "ribovlavin", "folate", "glutamate", 


https://www.malacards.org/card/graves_disease_5 has a paper that mentions butyrate. I mean, this is pretty surface level, and doesn't really get into any specific mechanisms or interactions (their "mined publications") is likely kinda bogus...
But "glutmate" has a hit in GAD2 - Glutamate Decarboxylase 2. IDK if it's meaningful. Seems more related to auto-immune diabetes, but some studies look at interaction bw diabetes and graves, so mention it. Idk.

https://docs.genecards.org/malacards/guide/key-features

Hmmm... this is kind of complicated. I think my takeaway though is that it could work. Like if we care about disease detection, we should probably also have some kind of disease database or literature search


In [84]:
import requests
import time

organism = "fpr"  # Change to your organism code

# Step 1: Get all pathway IDs
response = requests.get(f"https://rest.kegg.jp/list/pathway/{organism}")
pathway_ids = [line.split("\t")[0].replace("path:", "") 
               for line in response.text.strip().split("\n")]

In [85]:
len(pathway_ids)

104

In [88]:
response = requests.get("https://rest.kegg.jp/list/genome/216851")

# organisms = []

In [90]:
print(response.text)

T02591	fpr; Faecalibacterium prausnitzii L2-6
T02592	fpa; Faecalibacterium prausnitzii SL3/3
T06780	fpra; Faecalibacterium duncaniae A2165
T08154	fdu; Faecalibacterium duncaniae JCM 31915
T10210	ftk; Faecalibacterium taiwanense
T10369	fah; Faecalibacterium wellingii

